# Reference-Metric Applications

Reference metrics connect coordinate maps to the symbolic and generated infrastructure used by curvilinear NRPy calculations. A `ReferenceMetric` stores the numerical coordinates, the map to Cartesian coordinates, orthogonal scale factors, metric quantities, singular directions, and precompute-ready coordinate factors.

[Index](../index.ipynb) | Previous: [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb) | Next: [Basis Transforms](basis_transforms.ipynb)

## Why This Matters

Coordinate choices should match the symmetry of the problem. This notebook compares Cartesian and spherical reference metrics and shows the factors generated code needs.

## What You Will See

- Cartesian and spherical coordinate summaries.
- A determinant identity check.
- Precompute-ready reference-metric factors.

## Table of Contents

1. [Runtime context and imports](#Runtime-context-and-imports)
2. [Coordinate maps and scale factors](#Coordinate-maps-and-scale-factors)
3. [Metric determinant check](#Metric-determinant-check)
4. [Precompute-ready factors](#Precompute-ready-factors)
5. [Next notebooks](#Next-notebooks)

## Runtime Context and Imports

The import location is printed only as runtime context. The notebook uses the importable `nrpy` package and current reference-metric tools.

In [1]:
import nrpy
import nrpy.params as par
import nrpy.reference_metric as refmetric
import sympy as sp
from nrpy.infrastructures.BHaH.rfm_precompute import ReferenceMetricPrecompute


print("nrpy import succeeded")

nrpy import succeeded


## Coordinate Maps and Scale Factors

The same interface covers Cartesian, spherical, and stretched curvilinear coordinates. The compact summaries below show the numerical coordinates, Cartesian map, orthogonal scale factors, metric diagonals, singular directions, and representative Jacobian entries.

In [2]:
def short(expr, width=88):
    text = sp.sstr(expr)
    return text if len(text) <= width else text[: width - 3] + "..."


def short_list(values):
    return [short(value) for value in values]


def metric_summary(coord_system):
    metric = refmetric.ReferenceMetric(coord_system, enable_rfm_precompute=False)
    print(f"\n{coord_system}")
    print("  xx:", short_list(metric.xx))
    print("  xx_to_Cart:", short_list(metric.xx_to_Cart))
    print("  scale factors:", short_list(metric.scalefactor_orthog))
    print("  ghatDD diagonal:", short_list(metric.ghatDD[i][i] for i in range(3)))
    print("  ghatUU diagonal:", short_list(metric.ghatUU[i][i] for i in range(3)))
    print("  detgammahat:", short(metric.detgammahat))
    print("  radial-like directions:", metric.radial_like_dirns)
    print(
        "  Jacobian samples:",
        short_list(
            [
                metric.Jac_dUCart_dDrfmUD[0][0],
                metric.Jac_dUCart_dDrfmUD[0][1],
                metric.Jac_dUCart_dDrfmUD[1][0],
                metric.Jac_dUCart_dDrfmUD[2][2],
            ]
        ),
    )
    return metric


metrics = {
    coord_system: metric_summary(coord_system)
    for coord_system in [
        "Cartesian",
        "Spherical",
        "SinhSpherical",
        "SinhCylindrical",
    ]
}


Cartesian
  xx: ['xx0', 'xx1', 'xx2']
  xx_to_Cart: ['xx0', 'xx1', 'xx2']
  scale factors: ['1', '1', '1']
  ghatDD diagonal: ['1', '1', '1']
  ghatUU diagonal: ['1', '1', '1']
  detgammahat: 1
  radial-like directions: [0, 1, 2]
  Jacobian samples: ['1', '0', '0', '1']

Spherical
  xx: ['xx0', 'xx1', 'xx2']
  xx_to_Cart: ['xx0*sin(xx1)*cos(xx2)', 'xx0*sin(xx1)*sin(xx2)', 'xx0*cos(xx1)']
  scale factors: ['1', 'xx0', 'xx0*sin(xx1)']
  ghatDD diagonal: ['1', 'xx0**2', 'xx0**2*sin(xx1)**2']
  ghatUU diagonal: ['1', 'xx0**(-2)', '1/(xx0**2*sin(xx1)**2)']
  detgammahat: xx0**4*sin(xx1)**2
  radial-like directions: [0]
  Jacobian samples: ['sin(xx1)*cos(xx2)', 'xx0*cos(xx1)*cos(xx2)', 'sin(xx1)*sin(xx2)', '0']

SinhSpherical
  xx: ['xx0', 'xx1', 'xx2']
  xx_to_Cart: ['AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)*cos(xx2)/(exp(1/SINHW) - exp(-1/SINHW))', 'AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)*sin(xx2)/(exp(1/SINHW) - exp(-1/SINHW))', 'AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*c

  scale factors: ['AMPL*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)/(exp(1/SINHW) - exp(-1/SINHW))', 'AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))/(exp(1/SINHW) - exp(-1/SINHW))', 'AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)/(exp(1/SINHW) - exp(-1/SINHW))']
  ghatDD diagonal: ['AMPL**2*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)**2/(exp(1/SINHW) - exp(-1/SINH...', 'AMPL**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**2/(exp(1/SINHW) - exp(-1/SINHW))**2', 'AMPL**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**2*sin(xx1)**2/(exp(1/SINHW) - exp(-1/SINH...']
  ghatUU diagonal: ['(exp(1/SINHW) - exp(-1/SINHW))**2/(AMPL**2*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SI...', '(exp(1/SINHW) - exp(-1/SINHW))**2/(AMPL**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**2)', '(exp(1/SINHW) - exp(-1/SINHW))**2/(AMPL**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**2*sin(...']
  detgammahat: AMPL**6*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)**2*(exp(xx0/SINHW) - exp(-xx0/...
  radial-like directions: [0]
  Jacobian samples: ['AMPL*(e


SinhCylindrical
  xx: ['xx0', 'xx1', 'xx2']
  xx_to_Cart: ['AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))*cos(xx1)/(exp(1/SINHWRHO) - exp(-1/S...', 'AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))*sin(xx1)/(exp(1/SINHWRHO) - exp(-1/S...', 'AMPLZ*(exp(xx2/SINHWZ) - exp(-xx2/SINHWZ))/(exp(1/SINHWZ) - exp(-1/SINHWZ))']
  scale factors: ['AMPLRHO*(exp(xx0/SINHWRHO)/SINHWRHO + exp(-xx0/SINHWRHO)/SINHWRHO)/(exp(1/SINHWRHO) -...', 'AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))/(exp(1/SINHWRHO) - exp(-1/SINHWRHO))', 'AMPLZ*(exp(xx2/SINHWZ)/SINHWZ + exp(-xx2/SINHWZ)/SINHWZ)/(exp(1/SINHWZ) - exp(-1/SINH...']
  ghatDD diagonal: ['AMPLRHO**2*(exp(xx0/SINHWRHO)/SINHWRHO + exp(-xx0/SINHWRHO)/SINHWRHO)**2/(exp(1/SINHW...', 'AMPLRHO**2*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))**2/(exp(1/SINHWRHO) - exp(-1/SINH...', 'AMPLZ**2*(exp(xx2/SINHWZ)/SINHWZ + exp(-xx2/SINHWZ)/SINHWZ)**2/(exp(1/SINHWZ) - exp(-...']
  ghatUU diagonal: ['(exp(1/SINHWRHO) - exp(-1/SINHWRHO))**2/(AMPLRHO**2*(exp(xx0/SINHWRHO

## Metric Determinant Check

For an orthogonal reference metric, the determinant is the square of the product of the scale factors. The residual below checks that identity for spherical coordinates.

In [3]:
spherical = metrics["Spherical"]
scale_product = sp.prod(spherical.scalefactor_orthog)
determinant_residual = sp.trigsimp(
    sp.simplify(spherical.detgammahat - scale_product**2)
)
print("Spherical determinant residual:", determinant_residual)

Spherical determinant residual: 0


## Precompute-Ready Factors

Stretched coordinate systems contain repeated functions of individual numerical coordinates. RFM precompute extracts those factors so generated code can allocate, fill, and reuse them.

In [4]:
par.set_parval_from_str("parallelization", "openmp")
par.set_parval_from_str("fp_type", "double")
par.set_parval_from_str(
    "CoordSystem_to_register_CodeParameters", "SinhCylindrical"
)

precompute_metric = refmetric.reference_metric["SinhCylindrical_rfm_precompute"]
precompute = ReferenceMetricPrecompute("SinhCylindrical")

print("Precompute metric:", precompute_metric.CoordSystem)
print("Stored member count:", len(precompute.member_specs))
print("First member names:", [name for name, _ in precompute.member_specs[:6]])

Setting up reference_metric[SinhCylindrical_rfm_precompute]...


Precompute metric: SinhCylindrical
Stored member count: 7
First member names: ['f0_of_xx0', 'f0_of_xx0__D0', 'f0_of_xx0__DD00', 'f0_of_xx0__DDD000', 'f3_of_xx2', 'f3_of_xx2__D2']


## Next Notebooks

[Basis transforms](basis_transforms.ipynb) applies the Jacobians stored on reference metrics. [Curvilinear wave equation](../3-wave_equation/wave_equation_curvilinear.ipynb) uses reference metrics and precompute-ready factors in a generated wave-equation workflow.

## Where This Leads

- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb) reviews the prerequisite step.
- [Basis Transforms](basis_transforms.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.